In [ ]:
using PMDPs
using Test
using POMDPSimulators

using StaticArrays
using POMDPs
using MCTS, DiscreteValueIteration
using StatsBase
using Plots
using POMDPSimulators
using D3Trees
using POMDPPolicies
using POMDPLinter
using Random
using DataFrames
# using CSV
# using CSVFiles
using BSON
using POMDPSimulators

using DrWatson
using Distributions
using BeliefUpdaters

using Plots
using Latexify

# Plan

Graphs I want:

- big:
 - demand-revenue
 - n_iter-revenue %tohle asi ne.

- small:
 - demand-revenue
 - demand-utilization


In [ ]:
plot_size = ((1+sqrt(5))/2, 1).*100 .*3
label_demand = "Expected number of requests (?)"
label_demand = "Expected demand"

In [ ]:
# res = DrWatson.collect_results(joinpath(datadir(), "results", "linear_problem"));

In [ ]:
res_folder = joinpath(datadir(), "results", "linear_problem")
res_files = readdir(res_folder)
    
res = BSON.load(joinpath(res_folder,res_files[1]))

In [ ]:
function res2row(res)
    row_dfs = []
    
    # process pp_params into df with single row
    row = DataFrame(res[:pp_params])
    push!(row_dfs, row)
    
    # process agg: move evry row from "description" into single dataframe of one row
    agg = select(res[:agg], [:variable, :mean, :median, :min, :max])
    for row in eachrow(agg)
        v = string(row.variable)
        row = DataFrame(row[2:end])
        rename!(s-> s*"_$v", row)
        push!(row_dfs, row)
    end
    
    # compact dfs into single row
    df = hcat(row_dfs...)
    
    # add method name
    method = first(res[:results]).name
    insertcols!(df, 1, :method=>method)
end

res_folder = joinpath(datadir(), "results", "graph_problem")
res_files = readdir(res_folder)
    
res = BSON.load(joinpath(res_folder,res_files[1]))

res2row(res)

In [ ]:
res_folder = joinpath(datadir(), "results", "linear_problem")
res_files = readdir(res_folder)

rows = []
for res_file in res_files
    res = BSON.load(joinpath(res_folder,res_file))
    row = res2row(res)
    push!(rows, row)
end

results = vcat(rows...);
sort!(results, [:method, :expected_res]);

In [ ]:
results

In [ ]:
grps = groupby(results, :method);

In [ ]:
plot()
for gr in grps
    plot!(gr.expected_res, gr.mean_r, label=first(gr).method)
end
plot!()

In [ ]:
plot(edf1[!, x_param], edf1.r_mc,
    xlabel = label_demand,
    ylabel = "Utilization",
    label = "MCTS",
    m=(3, :circle), 
    size=plot_size,
    legend=:topleft
)
plot!(edf1[!, x_param], edf1.r_vi,  m=(3, :square, :black), line=(:black), label="VI")
plot!(edf1[!, x_param], edf1.r_flat,  m=(3, :dtriangle, :red), line=(:red), label="flat-rate")
plot!(edf1[!, x_param], edf1.r_hind,  m=(3, :utriangle, :green), line=(:green), label="hindsight")


# Old Code

In [ ]:
function get_result(df::DataFrame, c::Symbol, method::Symbol)
    [nt[method] for nt in df[!, c]]
end

function get_param(df::DataFrame, pt::Symbol, p::Symbol)
    [d[pt][p] for d in df[!, :params]]
end

# Experiment with varying n_iterations

In [ ]:
# mdp_dir = readdir(datadir("sims"))[2]
# exp = readdir(datadir("sims", mdp_dir))[1]
# # res = wload(datadir("sims", mdp_dir, exp))
# mdp_dir

# mdp_dir = readdir(datadir("sims"))[2]
# exp = readdir(datadir("sims", mdp_dir))[1]
# # res = wload(datadir("sims", mdp_dir, exp))

df = collect_results(
    datadir("sims", "exp_n_iter");
    white_list = [:r, :u, :t, :params], subfolders=true
)

In [ ]:
r_flat =  get_result(df, :r, :flat)
r_mc=get_result(df, :r, :mc)
r_hind =get_result(df, :r, :hind);

n_runs = get_param(df, :exp, :n_runs)
n_iterations = get_param(df, :mcts, :n_iterations) 
depth = get_param(df, :mcts, :depth);

mdf = DataFrame(@dict r_flat r_mc r_hind n_runs n_iterations depth )

sort!(mdf)

In [ ]:
edf1 = mdf
sort!(edf1)
x_param = :n_iterations
label_n_iter = "MCTS Iterations"

plot(edf1[!, x_param], edf1.r_mc,
    xlabel = label_n_iter,
    ylabel = "Revenue",
    label = "MCTS",
    m=(3, :circle), 
    size=plot_size,
    legend=:topleft
)
plot!(edf1[!, x_param], edf1.r_flat[1]*ones(length(edf1.r_mc)),  m=(3, :dtriangle), line=(:red), label="flat-rate")
plot!(edf1[!, x_param], edf1.r_hind[1]*ones(length(edf1.r_mc)),  m=(3, :utriangle), line=(:green), label="hindsight")

In [ ]:
save(datadir("exp_pro", "exp_big_niter.csv"), mdf)

In [ ]:
savefig(papersdir("paper_AAMAS2021_MCTS_pricing", "fig", "plt_big_niter.pdf"))

In [ ]:
filter(row -> row.n_iterations==50000, mdf)

# Varying demand - BIG

In [ ]:
df = collect_results( datadir("sims", "exp_demand");
    white_list = [:r, :u, :t, :params], subfolders=true)

In [ ]:
r_flat =  get_result(df, :r, :flat)
r_mc=get_result(df, :r, :mc)
r_hind =get_result(df, :r, :hind);

n_runs = get_param(df, :exp, :n_runs)
n_iterations = get_param(df, :mcts, :n_iterations) 
demand = map(sum, get_param(df, :mdp, :demand))

mdf = DataFrame(@dict r_flat r_mc r_hind n_runs n_iterations demand )
sort!(mdf)

In [ ]:
mdf = CSV.read(datadir("exp_pro", "exp_big_d_r.csv"), DataFrame; delim=";")

In [ ]:
edf1 = mdf
sort!(edf1)
x_param = :demand

plot(edf1[!, x_param], edf1.r_mc,
    xlabel = label_demand,
    ylabel = "Revenue",
    label = "MCTS",
    m=(3, :circle), 
    size=plot_size,
    legend=:topleft
)
plot!(edf1[!, x_param], edf1.r_flat,  m=(3, :dtriangle), line=(:red), label="flat-rate")
plot!(edf1[!, x_param], edf1.r_hind,  m=(3, :utriangle), line=(:green), label="hindsight")

In [ ]:
savefig(papersdir("paper_AAMAS2021_MCTS_pricing", "fig", "plt_big_d_r.pdf"))

# Varying demand - small - REV

In [ ]:
df_s_r = collect_results( datadir("sims", "exp_small_r");
    subfolders=true)

In [ ]:
df = df_s_r
r_flat =  get_result(df, :r, :flat)
r_mc=get_result(df, :r, :mc)
r_vi=get_result(df, :r, :vi)
r_hind =get_result(df, :r, :hind);

u_flat =  get_result(df, :u, :flat)
u_mc=get_result(df, :u, :mc)
u_vi=get_result(df, :u, :vi)
u_hind =get_result(df, :u, :hind);

n_runs = get_param(df, :exp, :n_runs)
n_iterations = get_param(df, :mcts, :n_iterations) 
demand = map(sum, get_param(df, :mdp, :demand))

mdf = DataFrame(@dict r_flat r_mc r_vi r_hind u_flat u_mc u_vi u_hind n_runs n_iterations demand )
sort!(mdf)

In [ ]:
mdf_old = CSV.read(datadir("exp_pro", "exp_small_d_r.csv"), DataFrame; delim=";")

In [ ]:
edf1 = mdf
sort!(edf1)
x_param = :demand


plot(edf1[!, x_param], edf1.r_mc,
    xlabel = label_demand,
    ylabel = "Revenue",
    label = "MCTS",
    m=(3, :circle), 
    size=plot_size,
    legend=:topleft
)
plot!(edf1[!, x_param], edf1.r_vi,  m=(3, :square, :black), line=(:black), label="VI")
plot!(edf1[!, x_param], edf1.r_flat,  m=(3, :dtriangle, :red), line=(:red), label="flat-rate")
plot!(edf1[!, x_param], edf1.r_hind,  m=(3, :utriangle, :green), line=(:green), label="hindsight")

# plot(edf1[!, x_param], edf1.u_mc,
#     label = "MCTS",
#     m=(3, :circle, :white), 
#     size=plot_size,
#     legend=:topleft, axis=:right
# )
# plot!(edf1[!, x_param], edf1.u_vi,  m=(3, :square, :white), line=(:black), label="VI")
# plot!(edf1[!, x_param], edf1.u_flat,  m=(3, :dtriangle, :white), line=(:red), label="flat-rate")
# plot!(edf1[!, x_param], edf1.u_hind,  m=(3, :utriangle, :white), line=(:green), label="hindsight")

In [ ]:
savefig(papersdir("paper_AAMAS2021_MCTS_pricing", "fig", "plt_small_d_r.pdf"))

# Varying demand - small - UTIL

In [ ]:
df_s_u = collect_results(
    datadir("sims", "exp_small_u"),
    subfolders=true
)

## boxplot

In [ ]:
df_s_r

rs_highest_demand = df_s_r.rs[5]
# order of values is mc, vi, hindsight
rs = DataFrame(transpose(hcat(rs_highest_demand...)), [:mc, :vi, :hind])

In [ ]:
df_s_r.flat

In [ ]:
names(df_s_r)

In [ ]:
# revenues for highest demand
rs_highest_demand = df_s_u.rs[5]
# order of values is mc, vi, hindsight
rs = DataFrame(transpose(hcat(rs_highest_demand...)), [:mc, :vi, :hind])

In [ ]:
save(datadir("exp_pro", "small_us.csv"), rs)

In [ ]:
using StatsPlots
#using CSVFiles

In [ ]:
boxplot(ones(100), rs.mc; label="MCTS", color=:blue, alpha=0.75, 
    legend=false, 
    ylabel="Revenue", 
    xtick = ([1,2,3],["MCTS","VI","hindsight"]), 
    size=plot_size
)
boxplot!(2*ones(100), rs.vi, label="VI", color=:black, alpha=0.75)
boxplot!(3*ones(100), rs.hind, color=:green, alpha=0.75, label="hindsight")

In [ ]:
savefig(papersdir("paper_AAMAS2021_MCTS_pricing", "fig", "plt_small_boxplot.pdf"))

## plot

In [ ]:
r_flat =  get_result(df, :r, :flat)
r_mc=get_result(df, :r, :mc)
r_vi=get_result(df, :r, :vi)
r_hind =get_result(df, :r, :hind);

u_flat =  get_result(df, :u, :flat)
u_mc=get_result(df, :u, :mc)
u_vi=get_result(df, :u, :vi)
u_hind =get_result(df, :u, :hind);

n_runs = get_param(df, :exp, :n_runs)
n_iterations = get_param(df, :mcts, :n_iterations) 
demand = map(sum, get_param(df, :mdp, :demand))

mdf = DataFrame(@dict r_flat r_mc r_vi r_hind u_flat u_mc u_vi u_hind n_runs n_iterations demand )
sort!(mdf)

In [ ]:
mdf = CSV.read(datadir("exp_pro", "exp_small_d_u.csv"), DataFrame; delim=";")

In [ ]:
edf1 = mdf
sort!(edf1)
x_param = :demand


plot(edf1[!, x_param], edf1.r_mc,
    xlabel = label_demand,
    ylabel = "Utilization",
    label = "MCTS",
    m=(3, :circle), 
    size=plot_size,
    legend=:topleft
)
plot!(edf1[!, x_param], edf1.r_vi,  m=(3, :square, :black), line=(:black), label="VI")
plot!(edf1[!, x_param], edf1.r_flat,  m=(3, :dtriangle, :red), line=(:red), label="flat-rate")
plot!(edf1[!, x_param], edf1.r_hind,  m=(3, :utriangle, :green), line=(:green), label="hindsight")

# plot(edf1[!, x_param], edf1.u_mc,
#     label = "MCTS",
#     m=(3, :circle, :white), 
#     size=plot_size,
#     legend=:topleft, axis=:right
# )
# plot!(edf1[!, x_param], edf1.u_vi,  m=(3, :square, :white), line=(:black), label="VI")
# plot!(edf1[!, x_param], edf1.u_flat,  m=(3, :dtriangle, :white), line=(:red), label="flat-rate")
# plot!(edf1[!, x_param], edf1.u_hind,  m=(3, :utriangle, :white), line=(:green), label="hindsight")

In [ ]:
savefig(papersdir("paper_AAMAS2021_MCTS_pricing", "fig", "plt_small_d_u.pdf"))

# Compare big and small rev

In [ ]:
using CSV

In [ ]:
c_0 = 10*30
mdf_big = CSV.read(datadir("exp_pro", "exp_big_d_r.csv"), DataFrame; delim=";")

mdf_big[!, "s/d"] = c_0 ./ mdf_big.demand ./10 .*(1+2+3+4+5+6+7+8+9+10)
mdf_big[!, "h/mc"] = mdf_big.r_hind ./ mdf_big.r_mc

In [ ]:
mdf_big[!, ["demand", "s/d", "h/mc"]]

In [ ]:
c_0 = 3*2
mdf_small = CSV.read(datadir("exp_pro", "exp_small_d_r.csv"), DataFrame; delim=";")
mdf_small[!, "s/d"] = c_0 ./ mdf_small.demand ./3 .* (1+2+3)
mdf_small[!, "h/mc"] = mdf_small.r_hind ./ mdf_small.r_mc
mdf_small[!, ["demand", "s/d", "h/mc"]]